# Offline Evaluation Analysis

This notebook analyzes saved workflow outputs. It does not rerun the LLM.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR / 'test_data' / 'multiclass_results.csv'
SUMMARY_PATH = BASE_DIR / 'test_data' / 'multiclass_summary.csv'

RESULTS_PATH, SUMMARY_PATH

In [ ]:
results = pd.read_csv(RESULTS_PATH)
results.head()

In [ ]:
results['category'].value_counts().sort_index()

In [ ]:
match_columns = [
    'match_extraction_success',
    'match_member_id',
    'match_diagnosis',
    'match_requested_service',
    'match_claim_amount',
    'match_member_exists',
    'match_balance_sufficient',
    'match_final_decision',
]

def true_rate(series):
    valid = series[series.isin(['TRUE', 'FALSE'])]
    if len(valid) == 0:
        return None
    return (valid == 'TRUE').mean() * 100

per_class = results.groupby('category')[match_columns].agg(lambda s: true_rate(s))
per_class = per_class.round(2)
per_class

In [ ]:
overall = pd.Series({col: true_rate(results[col]) for col in match_columns}, name='OVERALL').round(2)
overall

In [ ]:
ax = per_class['match_final_decision'].sort_values().plot(kind='barh', figsize=(9, 4), color='steelblue')
ax.set_title('Final Decision Accuracy by Class')
ax.set_xlabel('Accuracy (%)')
ax.set_ylabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = per_class.copy()

if HAS_SEABORN:
    plt.figure(figsize=(10, 5))
    sns.heatmap(plot_df, annot=True, cmap='Blues', vmin=0, vmax=100)
    plt.title('Per-Class Metric Accuracy (%)')
    plt.tight_layout()
    plt.show()
else:
    plot_df

In [ ]:
error_view = results.copy()
error_view['has_any_failure'] = error_view[match_columns].apply(lambda row: any(v == 'FALSE' for v in row), axis=1)
error_view.loc[error_view['has_any_failure'], ['email_id', 'category', 'actual_final_decision', 'expected_final_decision'] + match_columns]